# Nhánh ablation nâng cao — P3 / P4 / P5 / P6 / main (Kaggle, 30 epoch)

Bật/tắt phương pháp bằng cờ `RUN_*` ở Cell 2, chọn `FORGET_PCT` và `SEED`.
Mỗi run: 30 epoch (ngân sách Forget-MI) · per-batch step · chọn checkpoint bằng **S_val**
(validation, KHÔNG dùng test/retrained) · báo cáo cả **last(E29)** · eval trên **D_t_final** (75% test).

Đủ metric Forget-MI (1-Sim, MIA, Forget/Test x AUC/Macro-F1/**Pairwise-AUC**/F1) + extra.

**Ước tính thời gian (T4, 3% tham chiếu):** P3/P5 ~0.5-0.8h · P4 ~0.5-0.8h · P6 ~0.7-1.1h · main ~0.9-1.4h.
6% x~2, 10% x~3. Chọn account theo đó.

**Cell 4** re-eval og/re/baseline trên CUNG `D_t_final` - bắt buộc để so công bằng, KHONG ghép số cũ tính trên full test.


In [ ]:
# Cell 1: setup repo + deps
import os, subprocess
WORK_DIR = '/kaggle/working'
REPO_DIR = f'{WORK_DIR}/Forget-MI-LoKU'
REPO_URL = 'https://github.com/nhnhu146/Forget-MI-LoKU.git'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
assert os.path.exists('training/adv_common.py'), \
    'Chua thay training/adv_common.py - commit+push nhanh advanced len GitHub roi re-import notebook.'
subprocess.run(['pip', 'install', '-q', 'pydicom', 'scikit-image', 'scikit-learn',
                'pyyaml', 'wandb', 'seaborn==0.13.2'], check=True)
subprocess.run(['pip', 'install', '-q', 'transformers==4.38.0', 'peft==0.10.0',
                'accelerate==0.27.0'], check=True)
import torch
assert torch.cuda.is_available(), 'Bat GPU trong Kaggle Settings truoc khi chay.'
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('GPU   :', torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: CO CHON PHUONG PHAP + path discovery
import glob, os

# ===== BAT/TAT phuong phap (dat True cai muon chay) =====
RUN_P3   = True
RUN_P4   = False
RUN_P5   = False
RUN_P6   = False
RUN_MAIN = False

FORGET_PCT = 3        # 3 | 6 | 10
SEED       = 42

# P4: sweep k (giai doan 1). Vi du [5,10,15,20]. MAIN: ablation selector.
P4_K_SWEEP     = [15]
MAIN_SELECTORS = ['sval']          # ['sval','valce'] de ablation selector

assert FORGET_PCT in (3, 6, 10)

def find_dataset(*slugs):
    for slug in slugs:
        direct = f'/kaggle/input/{slug}'
        if os.path.isdir(direct):
            return direct
        hits = glob.glob(f'/kaggle/input/datasets/*/{slug}')
        if hits:
            return sorted(hits)[0]
    return None

def first_existing(root, relatives):
    for rel in relatives:
        p = os.path.join(root, rel)
        if os.path.exists(p):
            return p
    return None

DATA_ROOT = find_dataset('forget-mi-data')
MODELS_ROOT = find_dataset('forget-mi-models-full', 'forget-mi-models-v2', 'forget-mi-models')
assert DATA_ROOT and MODELS_ROOT, 'Add forget-mi-data va forget-mi-models(-full) vao Kaggle.'

base_hits = glob.glob(os.path.join(MODELS_ROOT, '**', 'training_original_model', 'pytorch_model.bin'), recursive=True)
gold_hits = glob.glob(os.path.join(MODELS_ROOT, '**', f'model_retrained_{FORGET_PCT}per', '**', 'pytorch_model.bin'), recursive=True)
BASE_MODEL = os.path.dirname(sorted(base_hits, key=len)[0]) if base_hits else None
GOLD_MODEL = os.path.dirname(sorted(gold_hits, key=len)[0]) if gold_hits else BASE_MODEL
TEXT_DIR = first_existing(DATA_ROOT, ['data/metadata', 'metadata'])
IMG_DIR  = first_existing(DATA_ROOT, ['data/img_data', 'img_data'])
FORGET_CSV  = f'./data_splits/forget_set_{FORGET_PCT}per.csv'
RESULTS_CSV = '/kaggle/working/results_advanced.csv'
HISTORY_CSV = f'/kaggle/working/perepoch_advanced_{FORGET_PCT}per_s{SEED}.csv'
OUTPUT_DIR  = f'/kaggle/working/advanced_output/{FORGET_PCT}per_s{SEED}'

for name, path in {'base model': BASE_MODEL, 'text metadata': TEXT_DIR,
                   'images': IMG_DIR, 'forget csv': FORGET_CSV}.items():
    assert path and os.path.exists(path), f'Missing {name}: {path}'

COMMON_OVR = {
    'forget_set_path': FORGET_CSV,
    'base_model_path': BASE_MODEL,
    'bert_pretrained_dir': BASE_MODEL,
    'retrained_model_path': GOLD_MODEL,
    'text_data_dir': TEXT_DIR,
    'img_data_dir': IMG_DIR,
    'output_dir': OUTPUT_DIR,
    'results_csv_path': RESULTS_CSV,
    'history_csv_path': HISTORY_CSV,
}
print('FORGET_PCT :', FORGET_PCT, '| SEED', SEED)
print('BASE_MODEL :', BASE_MODEL)
print('GOLD_MODEL :', GOLD_MODEL if gold_hits else 'N/A (CosSim vo hieu)')
print('RESULTS_CSV:', RESULTS_CSV)
print('Chay:', [n for n, f in [('P3', RUN_P3), ('P4', RUN_P4), ('P5', RUN_P5), ('P6', RUN_P6), ('MAIN', RUN_MAIN)] if f])


In [ ]:
# Cell 3: RUNNER CHIU LOI (overnight Save Version) - 1 method fail thi log & chay tiep
import os, subprocess, time
RUN_LOG = []

def run_method(script, run_id, extra_ovr=None):
    ovr = dict(COMMON_OVR); ovr['id'] = run_id
    if extra_ovr:
        ovr.update(extra_ovr)
    arg = ','.join(f'{k}={v}' for k, v in ovr.items())
    env = {**os.environ, 'PYTHONPATH': '.', 'WANDB_MODE': 'disabled'}
    cmd = ['python', script, '--config', 'config_advanced_kaggle.yaml',
           '--seed', str(SEED), '--fresh', '--override', arg]
    print(chr(10) + '=' * 70 + f'{chr(10)}RUN {run_id}{chr(10)}' + '=' * 70)
    t0 = time.time()
    try:
        subprocess.run(cmd, env=env, check=True)
        RUN_LOG.append((run_id, 'OK', round((time.time() - t0) / 3600, 2)))
    except subprocess.CalledProcessError as e:
        print(f'FAILED {run_id} rc={e.returncode} - BO QUA, chay method tiep theo')
        RUN_LOG.append((run_id, f'FAIL rc={e.returncode}', round((time.time() - t0) / 3600, 2)))

pct = f'{FORGET_PCT}per'
if RUN_P3:
    run_method('training/forgetmi_p3.py', f'p3_{pct}_s{SEED}')
if RUN_P4:
    for k in P4_K_SWEEP:
        run_method('training/forgetmi_p4.py', f'p4_k{k}_{pct}_s{SEED}', {'p4_stage1_epochs': k})
if RUN_P5:
    run_method('training/forgetmi_p5.py', f'p5_{pct}_s{SEED}')
if RUN_P6:
    run_method('training/forgetmi_p6.py', f'p6_{pct}_s{SEED}')
if RUN_MAIN:
    for sel in MAIN_SELECTORS:
        run_method('training/forgetmi_main.py', f'main_{sel}_{pct}_s{SEED}', {'selector': sel})

print(chr(10) + '=' * 70 + f'{chr(10)}TONG KET RUN:')
for rid, st, h in RUN_LOG:
    print(f'  [{st:>10}] {rid}  ({h}h)')


In [ ]:
# Cell 4: RE-EVAL og / re / baseline tren CUNG D_t_final (bat buoc de so cong bang)
# Bat co + dien path model. Baselines: (label, model_type, path).
#   model_type='pretrained' cho thu muc HF (og/re); 'state_dict' cho .pth (Forget-MI epoch).
RUN_EVAL_OG = False
RUN_EVAL_RE = False
EVAL_BASELINES = [
    # ('forgetmi', 'state_dict', '/kaggle/input/.../epoch_29/model_state_dict.pth'),
    # ('cfk',      'pretrained', '/kaggle/input/.../cfk_model'),
]

def eval_only(label, mtype, mpath, method='reference'):
    ovr = dict(COMMON_OVR)
    arg = ','.join(f'{k}={v}' for k, v in ovr.items())
    env = {**os.environ, 'PYTHONPATH': '.', 'WANDB_MODE': 'disabled'}
    cmd = ['python', 'training/forgetmi_eval_only.py', '--config', 'config_advanced_kaggle.yaml',
           '--seed', str(SEED), '--label', label, '--model_type', mtype,
           '--model_path', mpath, '--method', method, '--override', arg]
    print(chr(10) + 'eval-only:', label)
    try:
        subprocess.run(cmd, env=env, check=True)
    except subprocess.CalledProcessError as e:
        print(f'FAILED eval {label} rc={e.returncode} - BO QUA')

if RUN_EVAL_OG:
    eval_only('og', 'pretrained', BASE_MODEL, method='reference')
if RUN_EVAL_RE:
    eval_only('re', 'pretrained', GOLD_MODEL, method='reference')
for (lbl, mtype, mpath) in EVAL_BASELINES:
    eval_only(lbl, mtype, mpath, method='baseline')
print(chr(10) + 'DONE re-eval references (neu co bat).')


In [ ]:
# Cell 5: xem bang ket qua (+ nhac tai CSV tu Output)
import os, pandas as pd
if not os.path.exists(RESULTS_CSV):
    print('CHUA co', RESULTS_CSV, '- co the moi run deu fail. Xem TONG KET RUN o Cell 3.')
else:
    df = pd.read_csv(RESULTS_CSV)
    cols = ['method', 'checkpoint_kind', 'id', 'MIA', 'MIA_paper', '1_minus_Sim',
            'Forget_AUC', 'Forget_Macro_F1', 'Forget_Pairwise_AUC',
            'Test_AUC', 'Test_Macro_F1', 'Test_Pairwise_AUC',
            'forget_ce', 'test_ce', 'total_optimizer_steps', 'unlearn_core_hours']
    show = [c for c in cols if c in df.columns]
    pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
    print(f'{len(df)} dong trong {RESULTS_CSV}:')
    print(df[show].to_string(index=False))
    print(chr(10) + 'TAI VE: results_advanced.csv + perepoch_advanced_*.csv tu tab Output cua Save Version.')
